# Non-IID Brain Tumor MRI (Dirichlet α = 0.5, 3 clients, 5 rounds)

**Experiment A — GPIFL:** CE + grid-wise Perona–Malik PIDL, λ = 0.5, same grid/hparams as the main setup except λ.

**Experiment B — FedAvg baseline:** `regularizer_type=none` (CE only), same non-IID partition.

**Partitioning:** Dirichlet label skew (`partitioning=dirichlet`, `dirichlet_alpha=0.5`) via `FL_RUN_OVERRIDE` → `get_federated_data()`.

**Outputs:** per-variant `final_model.pth`, logs under `results_non_iid/…`, summary `non_iid_comparison.csv`, optional IID reference row from `results/brain_tumor_mri/3_clients/` if present.

| Step | Action |
|------|--------|
| 1 | Clone + install |
| 2 | Download Brain Tumor MRI (KaggleHub) |
| 3 | Run both FL experiments + `finalize_experiment()` |
| 4 | Build CSV + zip + Colab download |

## 1 — Clone repository and dependencies

In [1]:
GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'

import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
else:
    print('Repo already cloned. Pulling latest...')
    os.system(f'git -C {PROJECT_ROOT} pull')

if not PROJECT_ROOT.exists():
    raise RuntimeError(f'Clone failed: {GITHUB_REPO!r}')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Working dir: {Path.cwd()}')

Working dir: /content/medical_fl_pidl


In [2]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt
!python -c "import flwr, torch, kagglehub; print(f'flwr={flwr.__version__}  torch={torch.__version__}')"
print('Dependencies OK.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.7 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.20.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cry

## 2 — Download Brain Tumor MRI only

In [3]:
import kagglehub
from data.kaggle_loader import find_image_root

bt_path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
DATA_ROOT = find_image_root(bt_path)
print('ImageFolder root:', DATA_ROOT)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
[find_image_root] Found (named training split): 'Training'
  Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
ImageFolder root: /kaggle/input/brain-tumor-mri-dataset/Training


## 3 — Experiment configuration and run loop

In [4]:
import gc, json, os, time

import torch
from flwr.simulation import run_simulation

from federated.server_app import app as _server_app
from federated.client_app import app as _client_app

NUM_CLIENTS        = 3
NUM_SERVER_ROUNDS  = 5
LOCAL_EPOCHS       = 2
BATCH_SIZE         = 32
LEARNING_RATE      = 0.001
IMAGE_SIZE         = 224
RANDOM_SEED        = 42

PARTITIONING       = 'dirichlet'
DIRICHLET_ALPHA    = 0.5

FEATURE_LAYER      = 'layer2'
USE_GRID_LOSS      = True
GRID_SIZE          = 4
K_PM               = 1.0

SECAGG_MAX_WEIGHT = 1048575

RESULTS_BASE = PROJECT_ROOT / 'results_non_iid' / 'brain_tumor_mri' / f'dirichlet_a{DIRICHLET_ALPHA}_{NUM_CLIENTS}c'
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

# Two runs: (folder name, regularizer, lambda_pm)
RUNS = [
    ('gpifl_lambda05', 'perona_malik', 0.5),
    ('fedavg_no_pidl', 'none',           0.0),
]

IID_REF_SUMMARY = PROJECT_ROOT / 'results' / 'brain_tumor_mri' / '3_clients' / 'fl_summary.json'

print('Results base:', RESULTS_BASE)

Results base: /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
def run_one_variant(variant_dir_name: str, reg_type: str, lam: float, resume: bool = True) -> dict:
    from federated.server_app import finalize_experiment

    log_dir = RESULTS_BASE / variant_dir_name
    st = dict(variant=variant_dir_name, log_dir=str(log_dir), skipped=False, success=False)

    if resume and (log_dir / 'fl_summary.json').exists():
        print(f'  [SKIP] {variant_dir_name} — fl_summary.json exists')
        st['skipped'] = st['success'] = True
        return st

    log_dir.mkdir(parents=True, exist_ok=True)
    recon = max(1, NUM_CLIENTS - 1)

    run_cfg = {
        'dataset_name':                    'brain_tumor_mri',
        'data_root':                       str(DATA_ROOT),
        'num_classes':                     '0',
        'num_clients':                     str(NUM_CLIENTS),
        'min_fit_clients':                 str(NUM_CLIENTS),
        'num_server_rounds':               str(NUM_SERVER_ROUNDS),
        'local_epochs':                    str(LOCAL_EPOCHS),
        'batch_size':                      str(BATCH_SIZE),
        'learning_rate':                   str(LEARNING_RATE),
        'image_size':                      str(IMAGE_SIZE),
        'feature_layer':                   FEATURE_LAYER,
        'regularizer_type':                reg_type,
        'lambda_pm':                       str(lam),
        'use_grid_loss':                   str(USE_GRID_LOSS and reg_type != 'none').lower(),
        'grid_size':                       str(GRID_SIZE),
        'k':                               str(K_PM),
        'random_seed':                     str(RANDOM_SEED),
        'log_dir':                         str(log_dir),
        'partitioning':                    PARTITIONING,
        'dirichlet_alpha':                 str(DIRICHLET_ALPHA),
        'secagg_num_shares':               str(NUM_CLIENTS),
        'secagg_reconstruction_threshold': str(recon),
        'secagg_max_weight':               str(SECAGG_MAX_WEIGHT),
    }

    os.environ['FL_RUN_OVERRIDE'] = json.dumps(run_cfg)
    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    backend_cfg = {'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}}

    print(f'  [RUN ] {variant_dir_name} -> {log_dir}')
    t0 = time.time()
    try:
        run_simulation(
            server_app=_server_app,
            client_app=_client_app,
            num_supernodes=NUM_CLIENTS,
            backend_config=backend_cfg,
        )
        print(f'  [OK  ] {time.time() - t0:.0f}s')
        st['success'] = True
        finalize_experiment()
    except Exception as exc:
        print(f'  [FAIL] {exc}')
        import traceback; traceback.print_exc()
    finally:
        os.environ.pop('FL_RUN_OVERRIDE', None)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return st


for name, rt, lm in RUNS:
    print('\n===', name, '===')
    run_one_variant(name, rt, lm, resume=True)


            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 2, 'num_gpus': 0.5}}
DEBUG:flwr:Asyncio event loop already running.



=== gpifl_lambda05 ===
  [RUN ] gpifl_lambda05 -> /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/gpifl_lambda05
[Server] Device: cuda  |  Log dir: /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/gpifl_lambda05


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[task] Building federated data for 3 clients from: /kaggle/input/brain-tumor-mri-dataset/Training (partitioning=dirichlet)
[build_federated_dataloaders] Loading ImageFolder from '/kaggle/input/brain-tumor-mri-dataset/Training' …
  → 5,600 images  |  4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
  → Grayscale source detected. Images will be converted to 3-channel RGB.
  → Split: 4,480 train  |  1,120 test  (stratified, seed=42)
  → Partitioning (dirichlet): client 0: 1,280  |  client 1: 2,947  |  client 2: 253
[dataset_utils] Summary saved → /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/gpifl_lambda05/dataset_summary.json
[build_federated_dataloaders] Done. 3 client loaders + 1 test loader ready.

[Server] Dataset  : brain_tumor_mri  (4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary'])
[Server] Test set : 1120 samples


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

[Server] Saved final global weights -> /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/fedavg_no_pidl/final_model.pth
[Server Eval] Round   5 | Loss: 0.7445  Acc: 67.23%  N=1120


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 693.59s
INFO :      	History (loss, distributed):
INFO :      		round 1: 1.0113420282091417
INFO :      		round 2: 0.674300868170602
INFO :      		round 3: 0.6448682461466108
INFO :      		round 4: 0.710646882227489
INFO :      		round 5: 0.7444598470415388
INFO :      	History (loss, centralized):
INFO :      		round 0: 2.2010862554822648
INFO :      		round 1: 1.0113420282091414
INFO :      		round 2: 0.674300868170602
INFO :      		round 3: 0.6448682461466108
INFO :      		round 4: 0.710646882227489
INFO :      		round 5: 0.7444598470415388
INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 0.3080357142857143),
INFO :      	              (1, 0.5839285714285715),
INFO :      	              (2, 0.7),
INFO :      	              (3, 0.7017857142857142),
INFO :      	              (4, 0.7035714285714286),
INFO :      	       

Round   5 | Train Acc: 93.54%  Loss: 0.2273  PIDL: 0.000000 | Client Val Acc: 67.23%  Loss: 0.7445 |  Server Acc: 67.23% | Elapsed: 136.8s
  [OK  ] 720s

  EXPERIMENT SUMMARY
  Dataset:  brain_tumor_mri   Clients: 3   Rounds: 6
  Best Accuracy             0.7036  (round 4)
  Final Accuracy            0.6723
  Best Balanced Acc         0.7036  (round 4)
  Final Balanced Acc        0.6723
  Best Macro F1             0.6177  (round 4)
  Final Macro F1            0.5911
  Best ROC-AUC              0.9454  (round 2)
  Best PR-AUC               0.8485  (round 2)
  Final ECE                 0.1014
  Train time (total)        0.0000
  Infer time (total)        58.5600


[ExperimentLogger] All outputs written to: /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/fedavg_no_pidl
  fl_rounds.csv        (6 rounds)
  per_class_metrics.csv
  fl_eval.json
  fl_summary.json


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[finalize_experiment] Final model saved -> /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/fedavg_no_pidl/final_model.pth


## 4 — Summary CSV (non-IID + optional IID reference)

In [6]:
import pandas as pd
from IPython.display import display

rows = []
for name, _, _ in RUNS:
    p = RESULTS_BASE / name / 'fl_summary.json'
    if not p.exists():
        print('Missing:', p)
        continue
    s = json.loads(p.read_text(encoding='utf-8'))
    rows.append({
        'variant':          name,
        'alpha':            DIRICHLET_ALPHA,
        'best_accuracy':    s.get('best_accuracy'),
        'best_macro_f1':    s.get('best_macro_f1'),
        'final_ece':        s.get('final_ece'),
        'final_roc_auc':    s.get('final_roc_auc_macro'),
    })

if IID_REF_SUMMARY.exists():
    s0 = json.loads(IID_REF_SUMMARY.read_text(encoding='utf-8'))
    rows.append({
        'variant':          'iid_reference_notebook01_3c',
        'alpha':            None,
        'best_accuracy':    s0.get('best_accuracy'),
        'best_macro_f1':    s0.get('best_macro_f1'),
        'final_ece':        s0.get('final_ece'),
        'final_roc_auc':    s0.get('final_roc_auc_macro'),
    })
    print('Appended IID reference from', IID_REF_SUMMARY)
else:
    print('IID reference not found (run notebook 01 for brain_tumor / 3_clients or copy fl_summary.json).')

df = pd.DataFrame(rows)
csv_path = RESULTS_BASE / 'non_iid_comparison.csv'
df.to_csv(csv_path, index=False)
print('Wrote', csv_path)
display(df)

Appended IID reference from /content/medical_fl_pidl/results/brain_tumor_mri/3_clients/fl_summary.json
Wrote /content/medical_fl_pidl/results_non_iid/brain_tumor_mri/dirichlet_a0.5_3c/non_iid_comparison.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,variant,alpha,best_accuracy,best_macro_f1,final_ece,final_roc_auc
0,gpifl_lambda05,0.5,0.685714,0.600784,0.156295,0.954486
1,fedavg_no_pidl,0.5,0.703571,0.617728,0.101436,0.931757
2,iid_reference_notebook01_3c,NaN,0.952679,0.952381,0.026896,0.996375


## 5 — Zip `results_non_iid` and download (Colab)

In [7]:
import shutil

zip_stem = str(PROJECT_ROOT / 'results_non_iid_archive')
archive = shutil.make_archive(zip_stem, 'zip', PROJECT_ROOT, 'results_non_iid')
print('Created', archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not on Colab — zip saved at:', archive)

Created /content/medical_fl_pidl/results_non_iid_archive.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>